In [ ]:
import os
import subprocess

import cv2
from matplotlib import pyplot as plt

In [ ]:
augs = ['depth', 'rotation', 'sector_width', 'translation']

tgt = 'A2C'
batch = 0
frame_no = 3

remote_path = '/storage/talg/src/ultrapips/experiments/assets/data/CAMUS/aug'

In [ ]:
def get_storage_pod():
    cmd = "kubectl get pod -l app=storage-gateway -o jsonpath='{.items[?(@.status.phase==\"Running\")].metadata.name}'"
    return subprocess.check_output(cmd, shell=True).decode().split()[0]

get_storage_pod()

In [ ]:
pod_name = get_storage_pod()
path_to_ls = os.path.join(remote_path, 'A2C', 'batch_0')

res = subprocess.run([
    'kubectl', 'exec', pod_name, '--', 'bash', '-c', f'ls -lh {path_to_ls}'
], capture_output=True, text=True).stdout.split('\n')

In [ ]:
variants = {a: [] for a in augs}

for line in res:
    for aug in augs:
        if aug in line:
            variants[aug].append(line.split(aug)[-1].removeprefix('-'))

for aug in augs:
    variants[aug] = sorted(variants[aug], key=lambda x: float(x))

variants

In [ ]:
def storage2local(remote_obj, local_obj):
    # Get the running pod name
    pod_name = get_storage_pod()

    # Execute the copy
    cmd = ['kubectl', 'cp', f'{pod_name}:{remote_obj}', local_obj]
    
    # Capture stderr to filter out the 'tar' warning as your shell function does
    res = subprocess.run(cmd, capture_output=True, text=True)
    
    if res.stderr:
        filtered_err = "\n".join([line for line in res.stderr.splitlines() 
                                 if "tar: Removing leading" not in line])
        if filtered_err: print(filtered_err)
    
    return res.returncode

In [ ]:
REDO = True

path_to_dl = os.path.join(remote_path.replace('aug', 'raw'), tgt, f'batch_{batch}', 'images')
storage2local(
    os.path.join(path_to_dl, f'frame{frame_no:03d}.png'),
    f'original.png'
)
for aug, aug_variants in variants.items():
    for aug_power in aug_variants:
        path_to_dl = os.path.join(remote_path, tgt, f'batch_{batch}', f'{aug}-{aug_power}', 'images')

        if os.path.exists(f'{aug}-{aug_power}.png') and not REDO:
            continue

        storage2local(
            os.path.join(path_to_dl, f'frame{frame_no:03d}_aug0.png'),
            f'{aug}-{aug_power}.png'
        )

In [ ]:
orig = cv2.imread('original.png', cv2.IMREAD_GRAYSCALE)

for aug, aug_variants in variants.items():
    n = 6

    fig, axes = plt.subplots(1, n, figsize=(n * 3, 4), constrained_layout=True)
    axes[0].imshow(orig, cmap='gray')
    axes[0].set_ylabel(aug, fontsize=24)
    axes[0].set_xticks([])
    axes[0].set_yticks([])

    for i, aug_power in enumerate(aug_variants[:5], start=1):
        im = cv2.imread(f'{aug}-{aug_power}.png', cv2.IMREAD_GRAYSCALE)
        
        axes[i].imshow(im, cmap='gray')
        axes[i].axis('off')

plt.show()